# 02 · Deferrable loads → scheduler `Job`s

**Goal.** Build the paper's **three** deferrable load types as lists of `Job`s, all on the
same UTC hourly grid the scheduler runs on:

| load | source | offline? |
|------|--------|----------|
| **HVAC** | NREL ResStock (committed MA parquet) | ✅ real, offline |
| **EV**   | Caltech ACN-Data (live API) | guarded → labelled demo |
| **AI**   | Alibaba GPU v2020 trace (download) | guarded → labelled demo |

Every `Job` has: `job_id, energy_kwh, earliest_start, deadline, power_kw`. Two derived
quantities matter:

- **duration** $=\lceil \text{energy}/\text{power}\rceil$ whole hours it must run;
- **slack** $=(\text{deadline}-\text{earliest\_start})-\text{duration}$ = spare hours the
  scheduler can slide the job around in. **No slack, no savings.**

In [1]:
# --- standard setup for every notebook in this paper ------------------------
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")                 # find cals (in ../)
from dotenv import load_dotenv; load_dotenv("../.env")  # loads EIA_API_KEY if present
from cals import *          # FACTORS, carbon_intensity, Job, schedule, fifo_baseline, ...
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import nb_utils as U             # guarded data loaders + figure helper (see notebooks/nb_utils.py)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1 · HVAC (real NREL ResStock, offline)

Each hour of measured electric-HVAC energy becomes a one-hour job that may run
`flex_hours` earlier or later (a stand-in for thermal inertia / comfort slack). The HVAC
**deadline is an assigned knob** (`flex_hours`), not a hard physical constraint -- there is
no "must finish charging" moment like an EV has.

In [2]:
hvac_jobs, run_hours, hvac_df = U.load_hvac_jobs(flex_hours=6)   # bldg486202_MA_year
print("HVAC jobs:", len(hvac_jobs))
j = hvac_jobs[0]
print("example:", j.job_id, "| energy %.2f kWh | power %.2f kW | dur %dh | slack %dh"
      % (j.energy_kwh, j.power_kw, duration_h(j), slack_h(j)))

HVAC jobs: 8363
example: nrel_20190101T05 | energy 1.82 kWh | power 1.82 kW | dur 1h | slack 12h


## 2 · EV charging (guarded: Caltech ACN-Data)

`acn_to_jobs` maps each charging session to a job: energy = kWh delivered, window =
[plug-in, unplug], power = kWh / time-actually-charging. Real sessions need `acnportal` +
`ACN_API_TOKEN`; absent those we **skip gracefully** to a labelled demo set (same field
shape as the API, so the adapter is unchanged).

In [3]:
ev_jobs, ev_source = U.get_ev_jobs()
print("EV source:", ev_source, "| EV jobs:", len(ev_jobs))

EV: real ACN-Data unavailable (ModuleNotFoundError); using a labelled DEMO EV set. Install acnportal + set ACN_API_TOKEN for real sessions.


EV source: DEMO synthetic EV (offline) | EV jobs: 200


## 3 · AI compute (guarded: Alibaba GPU v2020)

> **Download the real trace** (for anything reported):
> `cluster-trace-gpu-v2020` from
> <https://github.com/alibaba/clusterdata/tree/master/cluster-trace-gpu-v2020>,
> and drop the `pai_job_table` CSV into `../data/ai/`. With no CSV present we use a
> labelled **synthetic Alibaba-shaped** frame so this notebook runs offline.

A batch/training job runs on some GPUs for a while and must finish, but not at a specific
hour -- so it is deferrable. **Two assumptions must be swept** (they are modeling knobs, not
measurements):

- `gpu_power_kw` — per-GPU draw (A100/V100 ≈ 0.3–0.4 kW). Energy scales linearly with it.
- `flex_hours` — how deferrable a batch job is; it sets the slack. The trace has **no real
  deadline**, so like HVAC this is an **assigned knob**.

In [4]:
ai_jobs, ai_parsed, ai_source = U.get_ai_jobs(gpu_power_kw=0.4, flex_hours=6)
print("AI source:", ai_source, "| AI jobs:", len(ai_jobs))
ai_parsed.head(3)

AI: real Alibaba GPU v2020 trace not found in data/ai/; using a labelled SYNTHETIC Alibaba-shaped trace.
AI source: DEMO synthetic Alibaba (offline) | AI jobs: 299


,job_id,submit,duration_h,num_gpus
0,j0,2019-06-17 18:02:10+00:00,1.021944,1.0
1,j1,2019-06-06 05:34:25+00:00,5.672778,2.0
2,j2,2019-06-03 04:27:59+00:00,3.876944,1.0


## 4 · Slack distributions

Slack is what the scheduler has to work with. HVAC jobs get a fixed 2·`flex_hours` window;
EV slack is driven by real dwell (charge fast, sit plugged in); AI slack is
duration + `flex_hours`.

In [ ]:
# CURATED FIGURE (restores figures/02_slack_histograms.png as committed in 98f159a).
# The three-panel default was uninformative: HVAC slack is a CONSTANT 12 h
# (= 2 x flex_hours) by construction and AI slack is a constant flex_hours, so two of
# the three panels were single degenerate bars carrying no information, and the EV
# panel was squashed flat by its 138 h tail. Only EV slack is an empirical
# distribution, and its load-bearing feature is the zero-slack spike -- the jobs that
# cannot be shifted at all. Plot that, and state in the CAPTION why HVAC is absent.
ev_slack = np.array([slack_h(j) for j in ev_jobs])
hvac_slack = np.array([slack_h(j) for j in hvac_jobs])
assert len(set(hvac_slack.tolist())) == 1, "HVAC slack should be constant by construction"
n_zero = int((ev_slack == 0).sum())
n_tail = int((ev_slack > 24).sum())
counts = [int((ev_slack == k).sum()) for k in range(25)]

fig, ax = plt.subplots(figsize=(7.2, 3.9))
ax.bar(range(25), counts, width=0.85, edgecolor="white",
       color=["#c1121f"] + ["#4878a8"] * 24)          # zero-slack bar called out in red
ax.set_xlim(-0.8, 25.2)
ax.set_xticks(range(0, 26, 2))
ax.set_xlabel("slack (hours)  =  dwell $-$ charge duration")
ax.set_ylabel("EV charging jobs")
ax.annotate(
    f"{n_zero:,} jobs ({100 * n_zero / len(ev_jobs):.0f}%) have ZERO slack\n"
    "— they cannot be shifted at all and\nemit at their plug-in hour regardless",
    xy=(0.45, counts[0]), xytext=(3.1, counts[0] * 0.99), fontsize=9, va="top",
    bbox=dict(boxstyle="round,pad=0.4", fc="#fdecec", ec="#c1121f", lw=1.2),
    arrowprops=dict(arrowstyle="-", color="#c1121f", lw=1.2))
ax.text(0.985, 0.74, f"{n_tail} jobs with slack > 24 h not shown\n"
                     f"(tail runs to {int(ev_slack.max())} h)",
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.6"))
fig.tight_layout(); U.savefig(fig, "02_slack_histograms.png"); plt.show()
print(f"EV slack: n={len(ev_jobs):,} median={np.median(ev_slack):.0f} h "
      f"zero-slack={n_zero:,} ({100 * n_zero / len(ev_jobs):.1f}%) "
      f">24h={n_tail} max={int(ev_slack.max())} h")
print(f"NOTE: median DWELL is {np.median(ev_slack + np.array([duration_h(j) for j in ev_jobs])):.0f} h; "
      "the 2 h figure is SLACK, not dwell.")

In [6]:
# one small summary table for the paper
summary = pd.DataFrame({
    name: {"count": len(jobs),
           "median energy kWh": np.median([j.energy_kwh for j in jobs]).round(2),
           "median power kW": np.median([j.power_kw for j in jobs]).round(2),
           "median duration h": int(np.median([duration_h(j) for j in jobs])),
           "median slack h": int(np.median([slack_h(j) for j in jobs]))}
    for name, jobs in loads.items()}).T
display(summary)

,count,median energy kWh,median power kW,median duration h,median slack h
HVAC (real),8363.0,0.42,0.42,1.0,12.0
EV (demo),200.0,16.52,4.95,4.0,3.0
AI (demo),299.0,3.00,1.60,3.0,6.0
